# Feature Engineering

Merge the cleaned listings with summary features from the raw calendar and reviews data, then save a compact engineered dataset.

In [7]:
import pandas as pd

# Load data
listings = pd.read_csv("../data/processed/cleaned_listings.csv")
reviews = pd.read_csv(r"C:\Users\thoka\Downloads\Airbnb project\data\raw\reviews.csv")
calendar = pd.read_csv(r"C:\Users\thoka\Downloads\Airbnb project\data\raw\calendar.csv")


Merge datasets

In [11]:
# Start from listing-level data and only merge listing-level aggregates.
df = listings.copy()

calendar_small = calendar[['listing_id', 'available', 'price']].copy()
calendar_small['available_flag'] = (calendar_small['available'] == 't').astype('int8')
calendar_small['price_num'] = pd.to_numeric(
    calendar_small['price'].astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False),
    errors='coerce'
)

calendar_agg = (
    calendar_small
    .groupby('listing_id', as_index=False)
    .agg(
        avg_daily_price=('price_num', 'mean'),
        median_daily_price=('price_num', 'median'),
        total_days=('available_flag', 'size'),
        available_days=('available_flag', 'sum')
    )
)
calendar_agg['availability_rate'] = calendar_agg['available_days'] / calendar_agg['total_days']
calendar_agg['unavailable_days'] = calendar_agg['total_days'] - calendar_agg['available_days']

reviews_agg = (
    reviews[['listing_id', 'date']]
    .assign(date=pd.to_datetime(reviews['date'], errors='coerce'))
    .groupby('listing_id', as_index=False)
    .agg(
        review_count=('date', 'size'),
        first_review_date=('date', 'min'),
        last_review_date=('date', 'max')
    )
)

df = pd.merge(
    df,
    calendar_agg,
    left_on='id',
    right_on='listing_id',
    how='left',
    validate='one_to_one'
).drop(columns=['listing_id'], errors='ignore')

df = pd.merge(
    df,
    reviews_agg,
    left_on='id',
    right_on='listing_id',
    how='left',
    validate='one_to_one'
).drop(columns=['listing_id'], errors='ignore')

df['review_count'] = df['review_count'].fillna(0).astype('int32')

Create features

In [14]:
# Revenue potential
if 'avg_daily_price' in df.columns:
    price_col = 'avg_daily_price'
elif 'price' in df.columns:
    price_col = 'price'
else:
    raise KeyError("No usable price column found. Expected 'avg_daily_price' or 'price'.")

if 'availability_365' in df.columns:
    availability_col = 'availability_365'
elif 'available_days' in df.columns:
    availability_col = 'available_days'
else:
    raise KeyError("No usable availability column found. Expected 'availability_365' or 'available_days'.")

df['revenue'] = df[price_col] * df[availability_col]

# Price category (handles duplicated quantile edges safely)
df['price_category'] = pd.qcut(df[price_col], q=3, duplicates='drop')